# 02 — Data Cleaning
Cleaning pipeline: Deduplikasi, validasi Bounding Box (BBox) YOLO, penanganan NaN, dan filtering.

## Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Arahkan root path ke parent folder agar modul src dan config terbaca
sys.path.insert(0, str(Path.cwd().parent))

from config.paths import DATASETS, OUTPUT_DIR
from src.data_loader import YOLOLoader
from src.data_cleaner import DataCleaner

# Konfigurasi penamaan file output secara dinamis
CLEAN_CSV = OUTPUT_DIR / "cleaned_human_fall.csv"
CLEAN_PARQUET = OUTPUT_DIR / "cleaned_human_fall.parquet"

print("[INFO] Modul Data Cleaner berhasil dimuat. Siap membedah data!")


[INFO] Modul Data Cleaner berhasil dimuat. Siap membedah data!


## 1. Load Raw Data

In [3]:
# ==========================================
# CELL 2: LOAD RAW DATA (YOLO FORMAT)
# ==========================================
from src.data_loader import load_all_datasets

print("=== TAHAP 1: MEMUAT DATA MENTAH ===")

# Kita gunakan fungsi mesin global agar otomatis membaca folder train dan valid
# dari seluruh dataset yang terdaftar di config/paths.py
df_raw = load_all_datasets(DATASETS)

print(f"\n[INFO] Data Mentah (Raw) berhasil dimuat: {len(df_raw):,} baris.")
df_raw.head()

=== TAHAP 1: MEMUAT DATA MENTAH ===

📥 Inisialisasi Loader untuk: FALL_DETECTION


Memuat human_fall/train: 100%|██████████| 24188/24188 [10:49<00:00, 37.22it/s]


  ✓ Berhasil memuat 25,241 baris dari folder train.


Memuat human_fall/valid: 100%|██████████| 832/832 [00:28<00:00, 28.90it/s]


  ✓ Berhasil memuat 854 baris dari folder valid.


...

Memuat person_detection/valid: 100%|██████████| 352/352 [00:12<00:00, 28.67it/s]


  ✓ Berhasil memuat 1,137 baris dari folder valid.


Memuat person_detection/test: 100%|██████████| 102/102 [00:03<00:00, 28.32it/s]


  ✓ Berhasil memuat 266 baris dari folder test.

✅ PROSES SELESAI: Menghasilkan 146,974 kotak anotasi (BBox) dari 56,593 gambar unik.

[INFO] Data Mentah (Raw) berhasil dimuat: 146,974 baris.


,dataset,split,image_id,image_path,label_path,img_exists,img_width,img_height,line_no,class_id,class_name,bbox_x_center,bbox_y_center,bbox_width,bbox_height,format
0,human_fall,train,0_png.rf.9d8c0c3b2d7e0a9087209ab517f49e1e,d:\DBS Capstone Code\ds_workspace_2\safewatch-...,d:\DBS Capstone Code\ds_workspace_2\safewatch-...,True,640,640,1,2,Person,0.177344,0.388281,0.292285,0.636035,yolo
1,human_fall,train,0_png.rf.d0c7845e7ce7c59e9f385943a94b522b,d:\DBS Capstone Code\ds_workspace_2\safewatch-...,d:\DBS Capstone Code\ds_workspace_2\safewatch-...,True,640,640,1,2,Person,0.177344,0.388281,0.292285,0.636035,yolo
2,human_fall,train,0_png.rf.d180efef87d5e74b7bfe46de5ba258c5,d:\DBS Capstone Code\ds_workspace_2\safewatch-...,d:\DBS Capstone Code\ds_workspace_2\safewatch-...,True,640,640,1,2,Person,0.177344,0.388281,0.292285,0.636035,yolo
3,human_fall,train,100_png.rf.11548409330c6c7805f69303c21440dd,d:\DBS Capstone Code\ds_workspace_2\safewatch-...,d:\DBS Capstone Code\ds_workspace_2\safewatch-...,True,640,640,1,2,Person,0.310156,0.467969,0.173437,0.442285,yolo
4,human_fall,train,100_png.rf.363f983ca3395b0e5f9575d7eb2e8f7a,d:\DBS Capstone Code\ds_workspace_2\safewatch-...,d:\DBS Capstone Code\ds_workspace_2\safewatch-...,True,640,640,1,2,Person,0.310156,0.467969,0.173437,0.442285,yolo


## 2. Jalankan Cleaning Pipeline

In [4]:
print("=== TAHAP 2: PROSES PEMBERSIHAN (DATA CLEANING) ===")
print("Mengeksekusi pipeline: Hapus duplikat, filter Out-of-Bounds BBox, & Drop NaN...")

# Menjalankan pembersihan menggunakan class OOP milikmu
cleaner = DataCleaner(df_raw, format="yolo")
df_clean = cleaner.run()

print(f"[INFO] Data Bersih (Clean): {len(df_clean):,} baris.")
print(f"[INFO] Total data yang terbuang/cacat: {len(df_raw) - len(df_clean):,} baris.")

=== TAHAP 2: PROSES PEMBERSIHAN (DATA CLEANING) ===
Mengeksekusi pipeline: Hapus duplikat, filter Out-of-Bounds BBox, & Drop NaN...
🗑️ Dihapus 0 BBox karena file gambar tidak ditemukan.
🗑️ Dihapus 1 BBox yang duplikat secara identik.
🗑️ Dihapus 0 BBox dengan koordinat di luar batas (0-1).
📐 Koordinat BBox berhasil dikonversi ke Piksel.
🚀 Memulai ekstraksi MediaPipe (16 Fitur X,Y) dari Bounding Box...


Pose Extraction: 100%|██████████| 26987/26987 [42:23<00:00, 10.61it/s] 


✅ Ekstraksi selesai! Berhasil: 8034 pose. Dibuang: 18953 objek.
[INFO] Data Bersih (Clean): 8,034 baris.
[INFO] Total data yang terbuang/cacat: 138,940 baris.


## 3. Lihat Cleaning Report

In [5]:
print("=== TAHAP 3: LAPORAN AUDIT PEMBERSIHAN ===")

# Mengonversi dictionary report dari cleaner menjadi DataFrame agar tampil cantik di Jupyter
report_df = pd.DataFrame.from_dict(
    cleaner.get_report(), 
    orient="index", 
    columns=["Jumlah / Keterangan"]
)

# Menambahkan styling agar tabel terlihat rapi di Jupyter Notebook
display(report_df.style.set_caption("Tabel Ringkasan Pembersihan Data").set_table_styles([{
    'selector': 'caption',
    'props': [('font-size', '16px'), ('font-weight', 'bold')]
}]))

=== TAHAP 3: LAPORAN AUDIT PEMBERSIHAN ===


,Jumlah / Keterangan
Total Awal (Raw),8034
Total Akhir (Clean),8034
Rincian Log,"[{'step': 'remove_missing_images', 'removed': 0}, {'step': 'remove_duplicates', 'removed': 1}, {'step': 'filter_invalid_bbox', 'removed': 0}, {'step': 'mediapipe_extraction', 'success_pose': 8034, 'failed_pose_dropped': 138939}]"


## 4. Simpan Output

In [6]:
print("=== TAHAP 4: MENYIMPAN DATA BERSIH ===")

# 1. Simpan dalam format CSV (Untuk kemudahan dibaca manusia / Excel)
df_clean.to_csv(CLEAN_CSV, index=False)
print(f"[SUKSES] Tersimpan format CSV     -> {CLEAN_CSV.name}")

# 2. Simpan dalam format Parquet (Sangat ringan, kompresi tinggi, loading secepat kilat untuk AI)
# Pastikan library 'pyarrow' sudah terinstal di requirements.txt!
df_clean.to_parquet(CLEAN_PARQUET, index=False)
print(f"[SUKSES] Tersimpan format Parquet -> {CLEAN_PARQUET.name}")

print("\nData ini sekarang sudah suci dan siap divisualisasikan pada tahap EDA (03_eda_visualization.ipynb)!")

=== TAHAP 4: MENYIMPAN DATA BERSIH ===
[SUKSES] Tersimpan format CSV     -> cleaned_human_fall.csv
[SUKSES] Tersimpan format Parquet -> cleaned_human_fall.parquet

Data ini sekarang sudah suci dan siap divisualisasikan pada tahap EDA (03_eda_visualization.ipynb)!
